# Chapter 10: Probabilistic Machine Learning and Bayesian Inference
## Practical: Gaussian Process Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# Generate synthetic data: y = sin(2*pi*x) + noise
np.random.seed(42)
X_train = np.random.uniform(0, 1, size=20).reshape(-1, 1)
y_train = np.sin(2 * np.pi * X_train.ravel()) + 0.2 * np.random.randn(20)

# Define kernel: RBF + white noise
kernel = 1.0 * RBF(length_scale=0.1) + WhiteKernel(noise_level=0.2)
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.0, n_restarts_optimizer=10)
gp.fit(X_train, y_train)

X_test = np.linspace(0, 1, 100).reshape(-1, 1)
y_mean, y_std = gp.predict(X_test, return_std=True)

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, c='red', label='Training data', s=50)
plt.plot(X_test, y_mean, 'b-', label='GP mean', linewidth=2)
plt.fill_between(X_test.ravel(), y_mean - 2*y_std, y_mean + 2*y_std,
                 alpha=0.3, color='blue', label='2-sigma uncertainty')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Gaussian Process Regression', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimised kernel: {gp.kernel_}")

# Extension: Bayesian Linear Regression from Scratch

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Generate data with noise
X = np.random.uniform(-3, 3, size=30).reshape(-1, 1)
y = 2 * X.ravel()**2 - 0.5 * X.ravel() + 0.3 * np.random.randn(30)

# Polynomial features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
Phi = poly.fit_transform(X)

# Prior: zero mean, identity covariance
alpha = 1.0
beta = 1.0
S0_inv = alpha * np.eye(Phi.shape[1])
SN_inv = S0_inv + beta * Phi.T @ Phi
SN = np.linalg.inv(SN_inv)
mu_N = beta * SN @ Phi.T @ y

# Predictive variance for new points
X_test = np.linspace(-3, 3, 100).reshape(-1, 1)
Phi_test = poly.transform(X_test)
y_pred = Phi_test @ mu_N
y_var = 1/beta + np.diag(Phi_test @ SN @ Phi_test.T)
y_std = np.sqrt(y_var)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, c='red', label='Data', s=50)
plt.plot(X_test, y_pred, 'b-', label='Bayesian mean', linewidth=2)
plt.fill_between(X_test.ravel(), y_pred - 2*y_std, y_pred + 2*y_std,
                 alpha=0.3, color='blue', label='2-sigma uncertainty')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Bayesian Linear Regression (Quadratic Basis)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Observations

- GP mean accurately captures the sine function with only 20 noisy points.
- Uncertainty is small near training points and grows in regions without data.
- Kernel hyperparameters are automatically learned by maximising the marginal likelihood.
- Bayesian linear regression provides full posterior distributions and error bars.